<a href="https://colab.research.google.com/github/iangabby1/Colab-to-GitHub-Phase-3/blob/main/GoogleToGitHub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# 1. LOAD THE DATASET
filename = "/content/rusteze.xlsx"
print(f"Loading data from {filename}...")
df = pd.read_excel(filename, sheet_name='Modified')

# 2. DEFINE VARIABLES
groups = df['raceId']
X = df[['grid', 'constructorId', 'driverId', 'age', 'constructor_wins']]
y = df['points_modified']

# 3. SET UP CROSS-VALIDATION (80/20 Split equivalent)
gkf = GroupKFold(n_splits=5)

# 4. HYPERPARAMETER TUNING SETUP
print("\nSetting up Hyperparameter Tuning...")

# Random Forest parameters to test
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

# XGBoost parameters to test
xgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

# 5. TRAIN, TUNE, AND EVALUATE FUNCTION
def tune_evaluate_and_train(base_model, param_grid, model_name):
    print(f"\n--- Tuning & Evaluating {model_name} ---")

    # Initialize RandomizedSearchCV
    # n_iter=5 means it will randomly try 5 different combinations of hyperparameters to save time
    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_grid,
        n_iter=5,
        scoring='neg_mean_squared_error',
        cv=gkf,
        random_state=42,
        n_jobs=-1
    )

    # Fit the search to find the best parameters (passing 'groups' is crucial here to prevent leakage!)
    search.fit(X, y, groups=groups)

    best_model = search.best_estimator_
    print(f"Hyperparameters used: {search.best_params_}")

    # Now evaluate the best model using Cross-Validation to get our metrics
    mse_scores = []
    mae_scores = []
    r2_scores = []
    last_y_test = None
    last_preds = None

    for fold, (train_index, test_index) in enumerate(gkf.split(X, y, groups)):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        best_model.fit(X_train, y_train)
        preds = best_model.predict(X_test)

        mse_scores.append(mean_squared_error(y_test, preds))
        mae_scores.append(mean_absolute_error(y_test, preds))
        r2_scores.append(r2_score(y_test, preds))
        last_y_test = y_test # Store for plotting
        last_preds = preds # Store for plotting

    print(f"\n{model_name} Final Evaluation Metrics:")
    print(f"Average MSE: {np.mean(mse_scores):.2f}")
    print(f"Average R²:  {np.mean(r2_scores):.2f}")
    print(f"Average MAE: {np.mean(mae_scores):.2f} positions")

    # Retrain on the entire dataset for final deployment
    print(f"\nTraining final {model_name} on the full dataset...")
    best_model.fit(X, y)

    # Save the trained model
    file_path = f"{model_name.lower().replace(' ', '_')}_f1_tuned_model.joblib"
    joblib.dump(best_model, file_path)
    print(f"Model saved to: {file_path}")

    return best_model, last_y_test, last_preds

# 6. EXECUTE
# Initialize base models
rf_base = RandomForestRegressor(random_state=42)
xgb_base = XGBRegressor(random_state=42, objective='reg:squarederror')

# Run Tuning and Evaluation
tuned_rf, rf_y_test, rf_preds = tune_evaluate_and_train(rf_base, rf_param_grid, "Random Forest")
tuned_xgb, xgb_y_test, xgb_preds = tune_evaluate_and_train(xgb_base, xgb_param_grid, "XGBoost")

print("\nFinished!")